# Region-wise Ensemble: V8.0 + V10.2

**Strategy:** Combine two complementary models using per-region weighted averaging of softmax probability maps.

| Model | Mean Dice | Text Delta | Strength |
|-------|-----------|------------|----------|
| **V8.0** (supervised pretrain) | 0.8753 | +0.00% | Strong absolute performance |
| **V10.2** (SSL pretrain + text) | ~0.84 | +0.77% | Positive text delta |

**Ensemble logic:**
- For each test case, both models produce 4-class softmax probability maps
- Per-class weights blend the two predictions: `prob = (1-w) * prob_A + w * prob_B`
- Different regions (ET, TC, WT) use different weights for model B
- Default: V8.0 dominates overall (w=0.3), V10.2 gets more weight on ET (w=0.5)

**Goal:** Achieve both high absolute Dice AND meaningful text delta.

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi 2>/dev/null || echo 'No GPU'
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Clone or pull repo
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0: break
        if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else: raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS data
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)

# Verify checkpoints exist on Drive
CKPT_A = os.path.join(DRIVE_CKPT, 'best_V8.0.pth')
CKPT_B = os.path.join(DRIVE_CKPT, 'best_V10.2.pth')
for name, path in [('V8.0', CKPT_A), ('V10.2', CKPT_B)]:
    assert os.path.exists(path), f'Checkpoint not found: {path}'
    sz = os.path.getsize(path) / 1024 / 1024
    print(f'  {name}: {path} ({sz:.1f} MB)')

print('Setup complete')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sun Apr 12 09:09:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|          

In [ ]:
# ===== Ensemble Evaluation (default weights) =====
import subprocess, os
os.chdir(REPO_DIR)

CONFIG_A = 'configs/autoresearch/V8.0_stage2_finetune.yaml'
CONFIG_B = 'configs/autoresearch/V10.2_ssl_pretrain.yaml'

# Default weights: V8.0 dominates, V10.2 gets more say on ET
W_ET = 0.5   # ET: 50% V10.2 (leveraging its text delta on ET)
W_TC = 0.3   # TC: 30% V10.2
W_WT = 0.3   # WT: 30% V10.2

cmd = [
    'python', '-u', 'scripts/ensemble_evaluate.py',
    '--ckpt-a', CKPT_A,
    '--ckpt-b', CKPT_B,
    '--config-a', CONFIG_A,
    '--config-b', CONFIG_B,
    '--weights-et', str(W_ET),
    '--weights-tc', str(W_TC),
    '--weights-wt', str(W_WT),
    '--split', 'test',
    '--overlap', '0.5',
    '--tta',
]
print('Command:', ' '.join(cmd))
print()

ret = subprocess.run(cmd, capture_output=False, text=True)
if ret.returncode != 0:
    print(f'Exit code: {ret.returncode}')

Command: python -u scripts/ensemble_evaluate.py --ckpt-a /content/drive/MyDrive/TextMamba3D/checkpoints/best_V8.0.pth --ckpt-b /content/drive/MyDrive/TextMamba3D/checkpoints/best_V10.2.pth --config-a configs/autoresearch/V8.0_stage2_finetune.yaml --config-b configs/autoresearch/V10.2_ssl_pretrain.yaml --weights-et 0.5 --weights-tc 0.3 --weights-wt 0.3 --split test --overlap 0.5 --tta



In [ ]:
# ===== Grid Search: Find Optimal Per-Region Weights =====
# Uses coarse grid (step=0.1) to sweep all weight combinations.
# For finer search, reduce --grid-step to 0.05 (but 21^3=9261 combos, much slower).
import subprocess, os
os.chdir(REPO_DIR)

CONFIG_A = 'configs/autoresearch/V8.0_stage2_finetune.yaml'
CONFIG_B = 'configs/autoresearch/V10.2_ssl_pretrain.yaml'

cmd = [
    'python', '-u', 'scripts/ensemble_evaluate.py',
    '--ckpt-a', CKPT_A,
    '--ckpt-b', CKPT_B,
    '--config-a', CONFIG_A,
    '--config-b', CONFIG_B,
    '--split', 'test',
    '--overlap', '0.5',
    '--tta',
    '--grid-search',
    '--grid-step', '0.1',
]
print('Command:', ' '.join(cmd))
print()
print('This will try 11^3 = 1331 weight combinations.')
print('Each combo runs full sliding-window inference x2 models x text/notext.')
print('Estimated time: ~2-4 hours on A100 (75 test cases).')
print()

ret = subprocess.run(cmd, capture_output=False, text=True)
if ret.returncode != 0:
    print(f'Exit code: {ret.returncode}')